In [27]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [28]:
# Load the CSV
wool_insulation = pd.read_csv('data/wool_insulation_sonoff.csv')

# Convert to time so it works with library functions
wool_insulation = wool_insulation.rename(columns={'last_changed': 'time'})

# Convert last_changed to datetime
wool_insulation['time'] = pd.to_datetime(wool_insulation['time'])

# Set time as index
wool_insulation = wool_insulation.set_index('time')

In [29]:
# Plot temperature data

# Map entity IDs to nicer names
entity_mapping = {
    'sensor.sonoff_snzb_02ld': 'Compost',
    'sensor.ewelink_snzb_02p_temperature': 'Outside',
}

wool_insulation['entity_name'] = wool_insulation['entity_id'].map(entity_mapping)

# Plot using the nicer names
fig = px.line(
    wool_insulation,
    y='state',
    color='entity_name',
    title='WiggleBin with wool insulation'
)

fig.update_layout(
    xaxis_title='Time',
    yaxis_title='Temperature (°C)',
    legend_title='Sensor'
)

fig.show()

In [30]:
# get historical weather data

# Load the CSV
historical_weather = pd.read_csv('data/2026_march_april_hourly_historical.csv')

# Convert to time so it works with library functions
historical_weather = historical_weather.rename(columns={'date': 'time'})

# Convert last_changed to datetime
historical_weather['time'] = pd.to_datetime(historical_weather['time'])

# Set time as index
historical_weather = historical_weather.set_index('time')

historical_weather

,temperature_2m,soil_temperature_0_to_7cm,weather_code,shortwave_radiation_instant,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high
time,,,,,,,,
2026-02-28 22:00:00+00:00,6.15,7.25,1.0,0.000000,23.0,9.0,23.0,0.0
2026-02-28 23:00:00+00:00,5.15,6.75,0.0,0.000000,1.0,1.0,0.0,0.0
2026-03-01 00:00:00+00:00,4.65,6.50,0.0,0.000000,0.0,0.0,0.0,0.0
2026-03-01 01:00:00+00:00,4.30,6.00,3.0,0.000000,97.0,0.0,0.0,97.0
2026-03-01 02:00:00+00:00,4.10,5.70,0.0,0.000000,5.0,0.0,4.0,2.0
...,...,...,...,...,...,...,...,...
2026-04-16 17:00:00+00:00,14.90,15.65,2.0,126.088264,73.0,16.0,30.0,59.0
2026-04-16 18:00:00+00:00,14.05,15.00,2.0,46.175938,66.0,28.0,17.0,49.0
2026-04-16 19:00:00+00:00,12.30,13.55,1.0,0.000000,41.0,23.0,3.0,21.0


In [31]:
# have both at same interval

In [32]:
# see correlation between weather and sensor data

In [33]:
# Prepare data for correlation analysis

# Extract compost and outside temperatures from wool_insulation
compost_temp = wool_insulation[wool_insulation['entity_name'] == 'Compost'].copy()
compost_temp['state'] = pd.to_numeric(compost_temp['state'], errors='coerce')

outside_temp = wool_insulation[wool_insulation['entity_name'] == 'Outside'].copy()
outside_temp['state'] = pd.to_numeric(outside_temp['state'], errors='coerce')

# Convert historical weather numeric columns to numeric
historical_weather_numeric = historical_weather.copy()
for col in historical_weather_numeric.columns:
    historical_weather_numeric[col] = pd.to_numeric(historical_weather_numeric[col], errors='coerce')

# Remove cloud_cover and weather_code columns
columns_to_drop = [col for col in historical_weather_numeric.columns 
                   if 'cloud_cover' in col.lower() or 'weather_code' in col.lower()]
historical_weather_numeric = historical_weather_numeric.drop(columns=columns_to_drop)

# Align the time indices - resample to common frequency (1 hour)
compost_temp_hourly = compost_temp[['state']].resample('h').mean()
compost_temp_hourly.columns = ['bin_compost_temperature']

outside_temp_hourly = outside_temp[['state']].resample('h').mean()
outside_temp_hourly.columns = ['bin_outside_temperature']

historical_weather_hourly = historical_weather_numeric.resample('h').mean()

# Merge the datasets on time
merged_data = compost_temp_hourly.join([outside_temp_hourly, historical_weather_hourly], how='inner')


In [34]:
# Calculate correlation matrix
import numpy as np

correlation_matrix = merged_data.corr()

# Get correlations with bin_compost_temperature
bin_compost_correlations = correlation_matrix['bin_compost_temperature'].drop('bin_compost_temperature').sort_values(ascending=False)

print("Correlation with Bin Compost Temperature:")
print("=" * 50)
print(bin_compost_correlations)

# Visualize the full correlation matrix as a heatmap
import plotly.figure_factory as ff

# Round correlation values to 2 decimal places for annotation
correlation_values_rounded = np.round(correlation_matrix.values, 2)

fig = ff.create_annotated_heatmap(
    z=correlation_matrix.values,
    annotation_text=correlation_values_rounded,
    x=correlation_matrix.columns.tolist(),
    y=correlation_matrix.columns.tolist(),
    colorscale='RdBu',
    zmid=0,
    zmin=-1,
    zmax=1,
    showscale=True,
    reversescale=False
)

fig.update_layout(
    xaxis_title='Variables',
    yaxis_title='Variables',
    width=900,
    height=800,
)

fig.show()


Correlation with Bin Compost Temperature:
bin_outside_temperature        0.497291
soil_temperature_0_to_7cm      0.444925
temperature_2m                 0.366738
shortwave_radiation_instant   -0.075830
Name: bin_compost_temperature, dtype: float64


In [38]:
# Stage 1: Predict outside temperature from weather variables
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# Prepare data for Stage 1
# Features: all weather variables (excluding bin temperatures and shortwave_radiation_instant)
weather_features = [col for col in merged_data.columns if col not in ['bin_compost_temperature', 'bin_outside_temperature', 'shortwave_radiation_instant']]

X = merged_data[weather_features].copy()
y = merged_data['bin_outside_temperature'].copy()

# Add time-based features
X['hour_of_day'] = X.index.hour
X['day_of_year'] = X.index.dayofyear

# Add cyclical encoding for time features (sine/cosine transform)
# This helps the model understand that hour 23 and hour 0 are similar
X['hour_sin'] = np.sin(2 * np.pi * X['hour_of_day'] / 24)
X['hour_cos'] = np.cos(2 * np.pi * X['hour_of_day'] / 24)
X['day_sin'] = np.sin(2 * np.pi * X['day_of_year'] / 365)
X['day_cos'] = np.cos(2 * np.pi * X['day_of_year'] / 365)

# Drop the linear versions and keep only cyclical encodings
X = X.drop(columns=['hour_of_day', 'day_of_year'])

# Remove rows with NaN values
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]

all_features = list(X.columns)

print(f"Data shape: X={X.shape}, y={y.shape}")
print(f"Weather features used: {weather_features}")
print(f"Time features added: hour_sin, hour_cos, day_sin, day_cos")
print(f"Total features: {all_features}")

# Split data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


Data shape: X=(1101, 6), y=(1101,)
Weather features used: ['temperature_2m', 'soil_temperature_0_to_7cm']
Time features added: hour_sin, hour_cos, day_sin, day_cos
Total features: ['temperature_2m', 'soil_temperature_0_to_7cm', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']

Training set size: 880
Testing set size: 221


In [39]:
# Train Linear Regression model for outside temperature prediction
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

model_stage1 = LinearRegression()
model_stage1.fit(X_train, y_train)

# Make predictions
y_train_pred = model_stage1.predict(X_train)
y_test_pred = model_stage1.predict(X_test)

# Evaluate model performance
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("Stage 1: Outside Temperature Prediction Model")
print("=" * 60)
print(f"\nTraining Set Performance:")
print(f"  R² Score: {train_r2:.4f}")
print(f"  RMSE: {train_rmse:.4f} °C")
print(f"  MAE: {train_mae:.4f} °C")

print(f"\nTesting Set Performance:")
print(f"  R² Score: {test_r2:.4f}")
print(f"  RMSE: {test_rmse:.4f} °C")
print(f"  MAE: {test_mae:.4f} °C")

# Feature importance (coefficients)
print(f"\nFeature Coefficients:")
for feature, coef in zip(weather_features, model_stage1.coef_):
    print(f"  {feature}: {coef:.6f}")
print(f"  Intercept: {model_stage1.intercept_:.6f}")


Stage 1: Outside Temperature Prediction Model

Training Set Performance:
  R² Score: 0.6352
  RMSE: 3.3965 °C
  MAE: 2.4710 °C

Testing Set Performance:
  R² Score: 0.6475
  RMSE: 2.2665 °C
  MAE: 1.7850 °C

Feature Coefficients:
  temperature_2m: 1.388235
  soil_temperature_0_to_7cm: -0.518595
  Intercept: 35.295103


In [40]:
# Visualize model predictions vs actual values
fig = go.Figure()

# Add test set actual values
fig.add_trace(go.Scatter(
    x=y_test.index,
    y=y_test.values,
    mode='lines',
    name='Actual Outside Temperature',
    line=dict(color='blue', width=2)
))

# Add test set predictions
fig.add_trace(go.Scatter(
    x=y_test.index,
    y=y_test_pred,
    mode='lines',
    name='Predicted Outside Temperature',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title=f'Stage 1: Outside Temperature Prediction (Test Set R² = {test_r2:.4f})',
    xaxis_title='Time',
    yaxis_title='Temperature (°C)',
    hovermode='x unified',
    height=500,
    width=1000
)

fig.show()

# Create scatter plot of predicted vs actual
fig_scatter = px.scatter(
    x=y_test.values,
    y=y_test_pred,
    labels={'x': 'Actual Temperature (°C)', 'y': 'Predicted Temperature (°C)'},
    title='Actual vs Predicted Outside Temperature (Test Set)'
)

# Add diagonal line for perfect prediction
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
fig_scatter.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Perfect Prediction',
    line=dict(color='green', dash='dash')
))

fig_scatter.update_layout(
    height=600,
    width=700
)

fig_scatter.show()
